### Vector Database (Qdrant) Client & Schema Test

**IMPORTANT: 사전 준비 단계**

```bash
# 1. Docker 서비스 시작 (PostgreSQL & Qdrant)
docker compose up -d

# 2. 서비스 상태 확인
docker compose ps

# 3. FastAPI 서버 실행
uvicorn src.app.api.main:app --reload
```

Day 5 작업 중 Vector Database 통합 테스트

#### 테스트 대상 모듈
1. `src/app/vector_db/client.py` - QdrantClientWrapper
2. `src/app/vector_db/schema.py` - CollectionSchema & 초기화 함수들

#### 테스트 항목
1. Qdrant 서버 연결 및 Health Check
2. Collection 생성 및 삭제
3. Collection 정보 조회
4. Payload Index 생성
5. Schema 검증
6. Vector DB 초기화 워크플로우
7. 리소스 정리 (테스트 데이터 삭제)

In [1]:
import sys
import os
from pathlib import Path

# 프로젝트 루트를 Python 경로에 추가
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.app.vector_db.client import QdrantClientWrapper, get_qdrant_client
from src.app.vector_db.schema import (
    CollectionSchema,
    setup_collection,
    verify_collection_schema,
    initialize_vector_db,
)
from src.app.core.config import settings

print("✓ Setup complete")
print(f"Qdrant Host: {settings.QDRANT_HOST}")
print(f"Qdrant Port: {settings.QDRANT_PORT}")
print(f"Collection Name: {settings.QDRANT_COLLECTION_NAME}")
print(f"Vector Size: {settings.QDRANT_VECTOR_SIZE}")

✓ Setup complete
Qdrant Host: localhost
Qdrant Port: 6333
Collection Name: research_articles
Vector Size: 1536


### 1. Qdrant Server Health Check

Qdrant 서버가 정상적으로 실행 중인지 확인

In [2]:
# QdrantClientWrapper 인스턴스 생성
client = QdrantClientWrapper()

# Health check
health = client.health_check()

print("Qdrant Server Health Check:")
print("=" * 60)
print(f"Status: {health['status']}")
print(f"Connected: {health['connected']}")
print(f"Host: {health['host']}")
print(f"Port: {health['port']}")

if health['connected']:
    print(f"\n✅ Qdrant 서버 연결 성공!")
    print(f"Available Collections: {health.get('collections', [])}")
else:
    print(f"\n❌ Qdrant 서버 연결 실패!")
    print(f"Error: {health.get('error')}")

Qdrant Server Health Check:
Status: healthy
Connected: True
Host: localhost
Port: 6333

✅ Qdrant 서버 연결 성공!
Available Collections: ['research_articles']


### 2. Collection Schema 정보 조회

CollectionSchema 클래스에 정의된 스키마 정보 확인

In [3]:
schema_info = CollectionSchema.get_schema_info()

print("Collection Schema Information:")
print("=" * 60)
print(f"Collection Name: {schema_info['collection_name']}")
print(f"Vector Size: {schema_info['vector_size']}")
print(f"Distance Metric: {schema_info['distance_metric']}")

print("\nPayload Schema:")
for field, field_type in schema_info['payload_schema'].items():
    print(f"  - {field}: {field_type}")

print("\nPayload Indexes:")
for idx in schema_info['payload_indexes']:
    print(f"  - {idx['field']}: {idx['type']}")

Collection Schema Information:
Collection Name: research_articles
Vector Size: 1536
Distance Metric: Cosine

Payload Schema:
  - article_id: string (UUID)
  - title: string
  - summary: string
  - source_type: string
  - category: string
  - importance_score: float
  - collected_at: string (ISO timestamp)
  - metadata: object

Payload Indexes:
  - source_type: keyword
  - category: keyword
  - importance_score: float
  - collected_at: keyword


### 3. Collection 존재 여부 확인

테스트 시작 전 기존 컬렉션 상태 확인

In [4]:
collection_name = CollectionSchema.COLLECTION_NAME

exists = client.collection_exists(collection_name)

print(f"Collection '{collection_name}' exists: {exists}")

if exists:
    info = client.get_collection_info(collection_name)
    print(f"\n현재 Collection 정보:")
    print(f"  - Name: {info['name']}")
    print(f"  - Vector Size: {info['vector_size']}")
    print(f"  - Points Count: {info['points_count']}")
    print(f"  - Status: {info['status']}")
else:
    print(f"\n컬렉션이 존재하지 않습니다.")

Collection 'research_articles' exists: True

현재 Collection 정보:
  - Name: research_articles
  - Vector Size: 1536
  - Points Count: 0
  - Status: green


### 4. Collection 생성 테스트 (recreate=True)

기존 컬렉션 삭제 후 새로 생성

In [5]:
print("Collection Recreate Test:")
print("=" * 60)

# 기존 컬렉션 삭제 후 재생성
success = client.recreate_collection(
    collection_name=collection_name,
    vector_size=CollectionSchema.VECTOR_SIZE,
    distance=CollectionSchema.DISTANCE_METRIC,
)

if success:
    print(f"✅ Collection '{collection_name}' 재생성 성공!")
    
    # 재생성된 컬렉션 정보 확인
    info = client.get_collection_info(collection_name)
    if info:
        print(f"\n재생성된 Collection 정보:")
        print(f"  - Name: {info['name']}")
        print(f"  - Vector Size: {info['vector_size']}")
        print(f"  - Points Count: {info['points_count']}")
        print(f"  - Status: {info['status']}")
else:
    print(f"❌ Collection 재생성 실패")

Collection Recreate Test:
✅ Collection 'research_articles' 재생성 성공!

재생성된 Collection 정보:
  - Name: research_articles
  - Vector Size: 1536
  - Points Count: 0
  - Status: green


### 5. Payload Index 생성 테스트

필터링 성능 향상을 위한 인덱스 생성

In [6]:
print("Payload Index Creation Test:")
print("=" * 60)

# 각 인덱스 생성
for index_config in CollectionSchema.PAYLOAD_INDEXES:
    try:
        client.client.create_payload_index(
            collection_name=collection_name,
            field_name=index_config["field_name"],
            field_schema=index_config["field_schema"],
        )
        print(f"✅ Index created on '{index_config['field_name']}'")
    except Exception as e:
        print(f"⚠️ Failed to create index on '{index_config['field_name']}': {e}")

print("\n✓ Payload Index 생성 완료")

Payload Index Creation Test:
✅ Index created on 'source_type'
✅ Index created on 'category'
✅ Index created on 'importance_score'
✅ Index created on 'collected_at'

✓ Payload Index 생성 완료


### 6. Schema 검증 테스트

생성된 컬렉션이 예상된 스키마와 일치하는지 검증

In [7]:
print("Schema Verification Test:")
print("=" * 60)

verification = verify_collection_schema(client)

print(f"Exists: {verification['exists']}")
print(f"Schema Valid: {verification['schema_valid']}")

if verification['exists'] and verification['schema_valid']:
    print(f"\n✅ 스키마 검증 성공!")
    print(f"\nCollection Info:")
    info = verification['info']
    print(f"  - Name: {info['name']}")
    print(f"  - Vector Size: {info['vector_size']}")
    print(f"  - Points Count: {info['points_count']}")
    print(f"  - Status: {info['status']}")
else:
    print(f"\n❌ 스키마 검증 실패!")
    print(f"\nErrors:")
    for error in verification['errors']:
        print(f"  - {error}")

Schema Verification Test:
Exists: True
Schema Valid: True

✅ 스키마 검증 성공!

Collection Info:
  - Name: research_articles
  - Vector Size: 1536
  - Points Count: 0
  - Status: green


### 7. setup_collection() 함수 테스트

setup_collection() 함수로 컬렉션 + 인덱스 일괄 설정

In [8]:
print("setup_collection() Function Test:")
print("=" * 60)

# 먼저 기존 컬렉션 삭제
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)
    print(f"기존 컬렉션 '{collection_name}' 삭제됨\n")

# setup_collection() 실행
success = setup_collection(client, recreate=False)

if success:
    print(f"\n✅ setup_collection() 성공!")
    
    # 결과 확인
    info = client.get_collection_info(collection_name)
    if info:
        print(f"\nSetup된 Collection:")
        print(f"  - Name: {info['name']}")
        print(f"  - Vector Size: {info['vector_size']}")
        print(f"  - Points Count: {info['points_count']}")
        print(f"  - Status: {info['status']}")
else:
    print(f"\n❌ setup_collection() 실패")

setup_collection() Function Test:
기존 컬렉션 'research_articles' 삭제됨


✅ setup_collection() 성공!

Setup된 Collection:
  - Name: research_articles
  - Vector Size: 1536
  - Points Count: 0
  - Status: green


### 8. initialize_vector_db() 전체 워크플로우 테스트

애플리케이션 시작 시 호출되는 메인 진입점 함수 테스트

In [9]:
print("initialize_vector_db() Full Workflow Test:")
print("=" * 60)

# 먼저 기존 컬렉션 삭제 (clean state)
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)
    print(f"기존 컬렉션 '{collection_name}' 삭제됨\n")

# initialize_vector_db() 실행
success = initialize_vector_db(recreate=False)

if success:
    print(f"\n✅ initialize_vector_db() 성공!")
    print(f"\n전체 워크플로우:")
    print(f"  1. ✓ Qdrant 서버 연결")
    print(f"  2. ✓ Health Check")
    print(f"  3. ✓ Collection 생성")
    print(f"  4. ✓ Payload Indexes 생성")
    print(f"  5. ✓ Schema 검증")
    
    # 최종 상태 확인
    verification = verify_collection_schema()
    print(f"\n최종 검증 결과:")
    print(f"  - Exists: {verification['exists']}")
    print(f"  - Schema Valid: {verification['schema_valid']}")
    print(f"  - Errors: {verification['errors']}")
else:
    print(f"\n❌ initialize_vector_db() 실패")

initialize_vector_db() Full Workflow Test:
기존 컬렉션 'research_articles' 삭제됨


✅ initialize_vector_db() 성공!

전체 워크플로우:
  1. ✓ Qdrant 서버 연결
  2. ✓ Health Check
  3. ✓ Collection 생성
  4. ✓ Payload Indexes 생성
  5. ✓ Schema 검증

최종 검증 결과:
  - Exists: True
  - Schema Valid: True
  - Errors: []


### 9. Singleton Pattern 테스트

get_qdrant_client() 싱글톤 패턴 검증

In [10]:
print("Singleton Pattern Test:")
print("=" * 60)

# 여러 번 호출
client1 = get_qdrant_client()
client2 = get_qdrant_client()
client3 = get_qdrant_client()

# 같은 인스턴스인지 확인
print(f"client1 is client2: {client1 is client2}")
print(f"client2 is client3: {client2 is client3}")
print(f"client1 is client3: {client1 is client3}")

if client1 is client2 is client3:
    print(f"\n✅ Singleton 패턴 정상 작동!")
    print(f"모든 get_qdrant_client() 호출이 동일한 인스턴스를 반환합니다.")
else:
    print(f"\n❌ Singleton 패턴 문제 발견!")

Singleton Pattern Test:
client1 is client2: True
client2 is client3: True
client1 is client3: True

✅ Singleton 패턴 정상 작동!
모든 get_qdrant_client() 호출이 동일한 인스턴스를 반환합니다.


### 10. Context Manager 테스트

`with` 문으로 안전한 리소스 관리

In [11]:
print("Context Manager Test:")
print("=" * 60)

# Context manager 사용
with QdrantClientWrapper() as test_client:
    health = test_client.health_check()
    print(f"Inside context: Health status = {health['status']}")
    print(f"Inside context: Connected = {health['connected']}")

print(f"\n✓ Context manager 정상 종료")
print(f"(연결이 자동으로 닫힘)")

Context Manager Test:
Inside context: Health status = healthy
Inside context: Connected = True

✓ Context manager 정상 종료
(연결이 자동으로 닫힘)


### 11. 에러 처리 테스트

#### 11.1 중복 생성 방지

In [12]:
print("Error Handling Test: Duplicate Collection")
print("=" * 60)

# 컬렉션이 이미 존재하는 상태에서 다시 생성 시도
try:
    client.create_collection(
        collection_name=collection_name,
        vector_size=CollectionSchema.VECTOR_SIZE,
    )
    print(f"❌ 중복 생성이 허용됨 (예상하지 못한 동작)")
except ValueError as e:
    print(f"✅ 중복 생성 방지 정상 작동!")
    print(f"Error: {e}")
except Exception as e:
    print(f"⚠️ 예상치 못한 에러: {type(e).__name__}: {e}")

Collection 'research_articles' already exists


Error Handling Test: Duplicate Collection
✅ 중복 생성 방지 정상 작동!
Error: Collection 'research_articles' already exists


#### 11.2 존재하지 않는 컬렉션 조회

In [13]:
print("Error Handling Test: Non-existent Collection")
print("=" * 60)

fake_collection_name = "non_existent_collection_12345"

# 존재하지 않는 컬렉션 정보 조회
info = client.get_collection_info(fake_collection_name)

if info is None:
    print(f"✅ 존재하지 않는 컬렉션에 대해 None 반환 (정상)")
else:
    print(f"❌ 예상치 못한 결과: {info}")

# 존재하지 않는 컬렉션 삭제 시도
delete_result = client.delete_collection(fake_collection_name)

if not delete_result:
    print(f"✅ 존재하지 않는 컬렉션 삭제 시 False 반환 (정상)")
else:
    print(f"❌ 예상치 못한 결과: {delete_result}")

Collection 'non_existent_collection_12345' does not exist
Collection 'non_existent_collection_12345' does not exist


Error Handling Test: Non-existent Collection
✅ 존재하지 않는 컬렉션에 대해 None 반환 (정상)
✅ 존재하지 않는 컬렉션 삭제 시 False 반환 (정상)


### 12. FastAPI 서버 통합 테스트

FastAPI 서버의 lifespan에서 initialize_vector_db()가 호출되는지 확인

In [14]:
import requests

print("FastAPI Server Integration Test:")
print("=" * 60)

# Health check 엔드포인트 호출
try:
    response = requests.get("http://127.0.0.1:8000/health", timeout=5.0)
    
    if response.status_code == 200:
        print(f"✅ FastAPI 서버 정상 실행 중")
        print(f"Response: {response.json()}")
        
        # 서버 시작 시 Vector DB가 초기화되었는지 확인
        exists = client.collection_exists(collection_name)
        if exists:
            print(f"\n✅ Vector DB 초기화 확인됨")
            print(f"Collection '{collection_name}' exists: {exists}")
            
            info = client.get_collection_info(collection_name)
            print(f"\nCollection Info:")
            print(f"  - Vector Size: {info['vector_size']}")
            print(f"  - Points Count: {info['points_count']}")
            print(f"  - Status: {info['status']}")
        else:
            print(f"\n⚠️ Vector DB가 초기화되지 않았습니다.")
    else:
        print(f"⚠️ 서버 응답 코드: {response.status_code}")
        
except requests.exceptions.ConnectionError:
    print("❌ FastAPI 서버에 연결할 수 없습니다.")
    print("서버를 시작하세요: uvicorn src.app.api.main:app --reload")
except Exception as e:
    print(f"❌ 테스트 실패: {e}")

FastAPI Server Integration Test:
✅ FastAPI 서버 정상 실행 중
Response: {'status': 'healthy'}

✅ Vector DB 초기화 확인됨
Collection 'research_articles' exists: True

Collection Info:
  - Vector Size: 1536
  - Points Count: 0
  - Status: green


### 13. 전체 테스트 요약

In [15]:
print("\n" + "=" * 80)
print("✅ Vector Database (Qdrant) 테스트 완료")
print("=" * 80)
print("""
테스트 완료된 항목:
  1. ✓ Qdrant 서버 Health Check
  2. ✓ Collection Schema 정보 조회
  3. ✓ Collection 존재 여부 확인
  4. ✓ Collection 생성/재생성 (recreate)
  5. ✓ Payload Index 생성
  6. ✓ Schema 검증 (verify_collection_schema)
  7. ✓ setup_collection() 함수
  8. ✓ initialize_vector_db() 전체 워크플로우
  9. ✓ Singleton Pattern 검증
 10. ✓ Context Manager 패턴
 11. ✓ 에러 처리 (중복 생성, 존재하지 않는 컬렉션)
 12. ✓ FastAPI 서버 통합 테스트

모든 테스트가 정상적으로 완료되었습니다! 🎉
""")
print("=" * 80)


✅ Vector Database (Qdrant) 테스트 완료

테스트 완료된 항목:
  1. ✓ Qdrant 서버 Health Check
  2. ✓ Collection Schema 정보 조회
  3. ✓ Collection 존재 여부 확인
  4. ✓ Collection 생성/재생성 (recreate)
  5. ✓ Payload Index 생성
  6. ✓ Schema 검증 (verify_collection_schema)
  7. ✓ setup_collection() 함수
  8. ✓ initialize_vector_db() 전체 워크플로우
  9. ✓ Singleton Pattern 검증
 10. ✓ Context Manager 패턴
 11. ✓ 에러 처리 (중복 생성, 존재하지 않는 컬렉션)
 12. ✓ FastAPI 서버 통합 테스트

모든 테스트가 정상적으로 완료되었습니다! 🎉



### 14. 테스트 데이터 정리 (Cleanup)

⚠️ **주의**: 이 셀을 실행하면 테스트 중 생성된 모든 Vector DB 데이터가 삭제됩니다!

In [16]:
print("Test Data Cleanup:")
print("=" * 60)

# 테스트 컬렉션 삭제
if client.collection_exists(collection_name):
    print(f"Deleting collection '{collection_name}'...")
    success = client.delete_collection(collection_name)
    
    if success:
        print(f"✅ Collection '{collection_name}' 삭제 완료")
    else:
        print(f"❌ Collection 삭제 실패")
else:
    print(f"Collection '{collection_name}' 이미 존재하지 않음")

# 클라이언트 연결 종료
try:
    client.close()
    print(f"✅ Qdrant 클라이언트 연결 종료")
except Exception as e:
    print(f"⚠️ 연결 종료 중 에러: {e}")

# 최종 확인
print("\n" + "=" * 60)
print("✓ 테스트 데이터 정리 완료")
print("\nNote: FastAPI 서버를 다시 시작하면 initialize_vector_db()가")
print("      자동으로 호출되어 컬렉션이 다시 생성됩니다.")
print("=" * 60)

Test Data Cleanup:
Deleting collection 'research_articles'...
✅ Collection 'research_articles' 삭제 완료
✅ Qdrant 클라이언트 연결 종료

✓ 테스트 데이터 정리 완료

Note: FastAPI 서버를 다시 시작하면 initialize_vector_db()가
      자동으로 호출되어 컬렉션이 다시 생성됩니다.
